# 🏋️ Kronos Fine-Tune — XGBoost on 5-Year Candle Data

**What this trains:**  
A classifier that predicts the **next candle direction** (up / down) from Kronos technical features.  
The saved model replaces the hand-coded composite score in `KronosAgent`.

**Pipeline:**
```
5y OHLCV  →  rolling Kronos features  →  XGBoost  →  models/kronos_model.json
```

**Features used:**
- EMA ratios (price/ema9, ema9/ema21, ema21/ema50, ema50/sma200)
- RSI(14)
- ATR % of price
- CPR position (above/inside/below)
- Price vs support/resistance
- Volume ratio
- Candlestick pattern flags (one-hot)
- Trend label (bullish=1, bearish=-1, sideways=0)

**Label:** `1` if `close[t+1] > close[t]`, else `0`

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SYMBOL       = "BTCUSDT"   # Delta: BTCUSDT | NSE equity: RELIANCE
SOURCE       = "delta"     # delta | fyers
INTERVAL     = "5m"        # 1m 5m 15m 1h 4h 1d
YEARS        = 5           # years of history to fetch
TEST_SPLIT   = 0.2         # fraction held out for test
MODEL_DIR    = "models"    # where to save the trained model
MODEL_NAME   = f"kronos_{SYMBOL}_{INTERVAL}.json"  # XGBoost model filename

In [ ]:
import sys, os, json, asyncio
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

Path(MODEL_DIR).mkdir(exist_ok=True)
print("Root:", ROOT)
print("Model will be saved to:", Path(MODEL_DIR) / MODEL_NAME)

In [ ]:
# ── Step 1: Fetch 5 years of candles ─────────────────────────────────────────
# Delta MCP only returns 7 days per call — we loop in 7-day chunks

async def fetch_chunk(symbol, source, interval, from_dt, to_dt):
    if source == "delta":
        from backend.brokers.delta.source import DeltaSource
        from backend.config import settings
        src = DeltaSource(
            api_key=settings.DELTA_API_KEY,
            api_secret=settings.DELTA_API_SECRET,
            region=settings.DELTA_REGION,
        )
    else:
        from backend.brokers.fyers.source import FyersSource
        from backend.config import settings
        src = FyersSource(
            client_id=settings.FYERS_CLIENT_ID,
            access_token=settings.FYERS_ACCESS_TOKEN,
        )
    try:
        df = await src.fetch_ohlc(symbol=symbol, from_dt=from_dt, to_dt=to_dt, interval=interval)
        return df
    finally:
        await src.close()


async def fetch_all(symbol, source, interval, years):
    """Fetch full history in 7-day chunks, concatenate."""
    end   = datetime.now(timezone.utc)
    start = end - timedelta(days=365 * years)
    chunk_days = 7
    all_dfs = []

    current = start
    total_chunks = int((end - start).days / chunk_days) + 1
    print(f"Fetching {years}y of {symbol} {interval} in {total_chunks} chunks...")

    while current < end:
        chunk_end = min(current + timedelta(days=chunk_days), end)
        try:
            df = await fetch_chunk(symbol, source, interval, current, chunk_end)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"  ⚠ chunk {current.date()} failed: {e}")
        current = chunk_end

    if not all_dfs:
        raise RuntimeError("No data fetched — check API keys and symbol")

    full = pd.concat(all_dfs)
    full = full[~full.index.duplicated(keep="first")].sort_index()
    print(f"Total candles: {len(full)}  ({full.index[0]} → {full.index[-1]})")
    return full


raw_df = await fetch_all(SYMBOL, SOURCE, INTERVAL, YEARS)
raw_df.head()

In [ ]:
# ── Step 2: Save raw candles (cache to avoid re-fetching) ─────────────────────
cache_path = Path(MODEL_DIR) / f"{SYMBOL}_{INTERVAL}_raw.parquet"
raw_df.to_parquet(cache_path)
print(f"Cached to {cache_path}")

# To reload: raw_df = pd.read_parquet(cache_path)

In [ ]:
# ── Step 3: Extract Kronos features for every candle ─────────────────────────
# We use the same pure-Python helpers from KronosAgent,
# applied with a rolling window so every row gets its own feature vector.

from backend.agents.kronos_agent import (
    _ema, _sma, _rsi, _atr, _pivot_cpr, _support_resistance,
    _classify_candles, _volume_signal,
)

ALL_PATTERNS = [
    "doji", "hammer", "hanging_man", "inverted_hammer", "shooting_star",
    "bullish_marubozu", "bearish_marubozu", "spinning_top",
    "bullish_engulfing", "bearish_engulfing",
    "bullish_harami", "bearish_harami",
    "morning_star", "evening_star",
    "three_white_soldiers", "three_black_crows",
]

WINDOW = 60  # minimum lookback needed for all indicators

def candles_from_df_slice(df_slice):
    rows = df_slice.reset_index()
    return [
        {"open": float(r["open"]), "high": float(r["high"]),
         "low": float(r["low"]),  "close": float(r["close"]),
         "volume": float(r["volume"])}
        for _, r in rows.iterrows()
    ]


def extract_features(candles):
    """Return a flat feature dict for the LAST candle in the window."""
    closes  = [c["close"]  for c in candles]
    volumes = [c["volume"] for c in candles]
    price   = closes[-1]

    e9  = _ema(closes, 9)
    e21 = _ema(closes, 21)
    e50 = _ema(closes, 50)
    s200= _sma(closes, 200)

    def last(lst):
        for v in reversed(lst):
            if v is not None: return v
        return 0.0

    le9, le21, le50, ls200 = last(e9), last(e21), last(e50), last(s200)

    rsi  = _rsi(closes, 14) or 50.0
    atr  = _atr(candles, 14) or 0.0
    sr   = _support_resistance(candles, 20)
    cpr  = _pivot_cpr(candles[:-1])
    pats = set(_classify_candles(candles[-5:]))

    # Volume ratio
    avg_vol = (sum(volumes[-11:-1]) / 10) if len(volumes) >= 11 else 1
    vol_ratio = volumes[-1] / avg_vol if avg_vol else 1.0

    # CPR position
    tc = cpr.get("tc", price)
    bc = cpr.get("bc", price)
    if price > tc:   cpr_pos = 1
    elif price < bc: cpr_pos = -1
    else:            cpr_pos = 0

    # Price vs nearest support/resistance
    sup  = sr["support"][0]    if sr["support"]    else price * 0.99
    res  = sr["resistance"][0] if sr["resistance"] else price * 1.01
    dist_to_sup = (price - sup) / price  if price else 0
    dist_to_res = (res - price) / price  if price else 0

    # Trend
    if le9 > le21 > le50:   trend = 1
    elif le9 < le21 < le50: trend = -1
    else:                   trend = 0

    feat = {
        # EMA ratios — normalised, scale-invariant
        "price_ema9_ratio":   price / le9   if le9   else 1.0,
        "ema9_ema21_ratio":   le9 / le21    if le21  else 1.0,
        "ema21_ema50_ratio":  le21 / le50   if le50  else 1.0,
        "ema50_sma200_ratio": le50 / ls200  if ls200 else 1.0,
        # Momentum
        "rsi":       rsi,
        "atr_pct":   (atr / price * 100) if price else 0,
        # Structure
        "trend":     trend,
        "cpr_pos":   cpr_pos,
        "dist_sup":  dist_to_sup,
        "dist_res":  dist_to_res,
        # Volume
        "vol_ratio": min(vol_ratio, 10.0),  # cap outliers
        # Candle body/wick
        "body_pct":  abs(closes[-1] - candles[-1]["open"]) / (candles[-1]["high"] - candles[-1]["low"] + 1e-9),
    }

    # Pattern one-hot flags
    for p in ALL_PATTERNS:
        feat[f"pat_{p}"] = 1 if p in pats else 0

    return feat


# Build feature matrix with rolling window
print(f"Extracting features with window={WINDOW}...")
rows_data = []
closes_all = raw_df["close"].tolist()

for i in range(WINDOW, len(raw_df) - 1):  # -1 because we need next candle for label
    slice_df = raw_df.iloc[i - WINDOW : i + 1]
    candles  = candles_from_df_slice(slice_df)
    feat     = extract_features(candles)
    feat["label"] = 1 if closes_all[i + 1] > closes_all[i] else 0  # next candle up?
    feat["ts"]    = raw_df.index[i]  # timestamp for train/test split
    rows_data.append(feat)

feat_df = pd.DataFrame(rows_data)
print(f"Feature matrix: {feat_df.shape}  — label balance: {feat_df['label'].mean():.1%} up")
feat_df.head()

In [ ]:
# ── Step 4: Train / Test split (time-based — NO data leakage) ─────────────────
split_idx = int(len(feat_df) * (1 - TEST_SPLIT))
train_df  = feat_df.iloc[:split_idx]
test_df   = feat_df.iloc[split_idx:]

FEATURE_COLS = [c for c in feat_df.columns if c not in ("label", "ts")]

X_train, y_train = train_df[FEATURE_COLS].values, train_df["label"].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df["label"].values

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Train period: {train_df['ts'].iloc[0]} → {train_df['ts'].iloc[-1]}")
print(f"Test  period: {test_df['ts'].iloc[0]} → {test_df['ts'].iloc[-1]}")

In [ ]:
# ── Step 5: Train XGBoost ─────────────────────────────────────────────────────
try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    import xgboost as xgb

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print("Training complete.")

In [ ]:
# ── Step 6: Evaluate ──────────────────────────────────────────────────────────
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)

print(f"Test accuracy : {acc:.4f}")
print(f"ROC-AUC       : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["DOWN", "UP"]))

In [ ]:
# ── Step 7: Feature importance ────────────────────────────────────────────────
import matplotlib.pyplot as plt

importance = pd.Series(model.feature_importances_, index=FEATURE_COLS)
top20 = importance.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#0d1117")
ax.set_facecolor("#161b22")
top20[::-1].plot.barh(ax=ax, color="#58a6ff")
ax.set_title(f"Top 20 Features — {SYMBOL} {INTERVAL}", color="#c9d1d9")
ax.tick_params(colors="#8b949e")
for spine in ax.spines.values(): spine.set_edgecolor("#30363d")
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 8: Equity curve simulation ──────────────────────────────────────────
# Simple long-only strategy: enter when model says UP with prob > threshold
THRESHOLD = 0.55

test_closes = raw_df["close"].iloc[WINDOW + split_idx : WINDOW + split_idx + len(y_test) + 1].values

capital = 1.0
equity  = [1.0]
bh      = [1.0]  # buy-and-hold benchmark

for i in range(len(y_pred_prob)):
    ret = (test_closes[i + 1] / test_closes[i]) - 1 if i + 1 < len(test_closes) else 0
    if y_pred_prob[i] > THRESHOLD:
        capital *= (1 + ret)
    equity.append(capital)
    bh.append(bh[-1] * (1 + ret))

fig, ax = plt.subplots(figsize=(14, 4), facecolor="#0d1117")
ax.set_facecolor("#161b22")
ax.plot(equity, color="#3fb950", linewidth=1, label=f"ML Strategy (>{THRESHOLD:.0%})")
ax.plot(bh,     color="#58a6ff", linewidth=1, label="Buy & Hold", alpha=0.7)
ax.axhline(1.0, color="#30363d", linewidth=0.5)
ax.set_title(f"Equity Curve — {SYMBOL} {INTERVAL}", color="#c9d1d9")
ax.tick_params(colors="#8b949e")
ax.legend(facecolor="#161b22", labelcolor="#c9d1d9")
for spine in ax.spines.values(): spine.set_edgecolor("#30363d")
plt.tight_layout()
plt.show()

print(f"Strategy return: {(equity[-1]-1)*100:+.1f}%")
print(f"Buy&Hold return: {(bh[-1]-1)*100:+.1f}%")

In [ ]:
# ── Step 9: Save model + metadata ────────────────────────────────────────────
import json as _json

model_path = Path(MODEL_DIR) / MODEL_NAME
model.save_model(str(model_path))
print(f"Model saved: {model_path}")

meta = {
    "symbol":        SYMBOL,
    "source":        SOURCE,
    "interval":      INTERVAL,
    "feature_cols":  FEATURE_COLS,
    "window":        WINDOW,
    "train_rows":    int(len(X_train)),
    "test_rows":     int(len(X_test)),
    "accuracy":      round(float(acc), 4),
    "roc_auc":       round(float(auc), 4),
    "threshold":     THRESHOLD,
    "trained_at":    datetime.now(timezone.utc).isoformat(),
}
meta_path = Path(MODEL_DIR) / MODEL_NAME.replace(".json", "_meta.json")
meta_path.write_text(_json.dumps(meta, indent=2))
print(f"Metadata saved: {meta_path}")
print(_json.dumps(meta, indent=2))

In [ ]:
# ── Step 10: Verify model loads correctly (sanity check) ─────────────────────
import xgboost as xgb

loaded = xgb.XGBClassifier()
loaded.load_model(str(model_path))

# Predict on last test sample
sample = X_test[-1:]
prob   = loaded.predict_proba(sample)[0][1]
print(f"Loaded model prediction on last test sample: {prob:.4f}  ({'UP' if prob > THRESHOLD else 'DOWN'})")
print("✅ Model ready for KronosAgent")
print()
print(f"Deploy with:")
print(f"  cp {model_path} <project_root>/models/")
print(f"  # KronosAgent auto-loads models/kronos_{{SYMBOL}}_{{INTERVAL}}.json")